<a href="https://colab.research.google.com/github/mari-muthu-k/agent-to-production/blob/main/notebooks/day1/Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1 Demo: How LLMs work

Needs `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
!pip install -q tiktoken requests

In [3]:
import os
try:
    from google.colab import userdata
    for name in ["LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "GEMINI_API_KEY"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
except ImportError:
    pass

if os.environ.get("GEMINI_API_KEY") and not os.environ.get("LLM_BASE_URL"):
    os.environ["LLM_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai"   # Gemini

def auth_headers() -> dict:
    """GEMINI_API_KEY on a Google URL: x-goog-api-key (+ the Bearer form Gemini's OpenAI-compatible endpoint
    requires). Otherwise: Authorization: Bearer LLM_API_KEY."""
    gemini = os.environ.get("GEMINI_API_KEY")
    if gemini and "googleapis.com" in os.environ.get("LLM_BASE_URL", ""):
        return {"x-goog-api-key": gemini, "Authorization": f"Bearer {gemini}", "Content-Type": "application/json"}
    return {"Authorization": f"Bearer {os.environ.get('LLM_API_KEY', '')}", "Content-Type": "application/json"}

import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # GPT-4o-era tokenizer
print("tokenizer loaded")

tokenizer loaded


---
## D3.1 What the model actually receives

In [ ]:
# text = "Researchers trained a small code model."
# text = "Researchers trained a small code model. Researchers"
text = "Researchers trained a small code model. notsouncommontbh"
ids = enc.encode(text)
print("text:     ", text)
print("token IDs:", ids)
print("pieces:   ", [enc.decode([i]) for i in ids])
print(f"\n{len(text)} characters -> {len(ids)} tokens")

text:      Researchers trained a small code model. notsouncommontbh
token IDs: [123315, 22203, 261, 3291, 3490, 2359, 13, 625, 82, 1379, 2280, 645, 22631]
pieces:    ['Researchers', ' trained', ' a', ' small', ' code', ' model', '.', ' not', 's', 'oun', 'comm', 'ont', 'bh']

56 characters -> 13 tokens


Note : common words are single tokens; the space attaches to the word **after** it.

---
## D3.2 Same text, three tokenizers  ·

**Before running:** ask the room to predict in chat which row costs the most tokens.

In [ ]:
encoders = {name: tiktoken.get_encoding(name) for name in ["gpt2", "cl100k_base", "o200k_base"]}
samples = {
    "English":   "Researchers trained a small code model.",
    "AI jargon": "LoRA fine-tuning of a Mixture-of-Experts transformer",
    "Python":    "def forward(self, x):\n    return self.attn(x)",
    "Tamil":     "ஆராய்ச்சியாளர்கள் ஒரு சிறிய மாதிரியை பயிற்றுவித்தனர்.",
}
print(f"{'text':10}" + "".join(f"{n:>14}" for n in encoders))
for label, t in samples.items():
    print(f"{label:10}" + "".join(f"{len(e.encode(t)):>14}" for e in encoders.values()))

text                gpt2   cl100k_base    o200k_base
English                7             7             7
AI jargon             15            13            13
Python                19            13            13
Tamil                149            69            20


Takeaways to say out loud:
1. Newer tokenizers use **fewer tokens** for the same text (bigger vocabularies).
2. **Jargon and acronyms** split into pieces.
3. **Non-English text** costs far more for the same meaning: English-based cost estimates are wrong for Tamil or Hindi users.

Optional: show the AI-jargon split piece by piece.

In [ ]:
jargon = samples["AI jargon"]
for name, e in encoders.items():
    print(f"{name:12}", [e.decode([i]) for i in e.encode(jargon)])

---
## D3.3 Code is tokens too

In [ ]:
snippet = """def forward(self, x):
    attn = self.attn(x)
    return self.proj(attn)"""
ids = enc.encode(snippet)
print("token IDs:", ids)
print("pieces:   ", [enc.decode([i]) for i in ids])

token IDs: [1314, 6687, 2402, 11, 1215, 1883, 271, 1927, 77, 314, 1051, 80365, 77, 4061, 446, 271, 622, 1051, 3231, 73, 104754, 77, 8]
pieces:    ['def', ' forward', '(self', ',', ' x', '):\n', '   ', ' att', 'n', ' =', ' self', '.att', 'n', '(x', ')\n', '   ', ' return', ' self', '.pro', 'j', '(att', 'n', ')']


Point at: `def`, `return` and `self` are single tokens; indentation is its own tokens.

---
## D3.4 Why models miscount letters (needs the API)

In [5]:
word = "strawberry"
print("the model sees:", [enc.decode([i]) for i in enc.encode(word)])
try:
    import requests
    base = os.environ["LLM_BASE_URL"].rstrip("/")
    # Thinking off, to show the raw miscount (D3.4b turns it on). OpenRouter and Gemini spell it differently.
    no_thinking = {"reasoning": {"enabled": False}} if "openrouter.ai" in base else {"reasoning_effort": "none"}
    r = requests.post(base + "/chat/completions", timeout=60,
        headers=auth_headers(),
        json={"model": os.environ["LLM_MODEL"].split(",")[0].strip(), "max_tokens": 20, "temperature": 0,
              **no_thinking,
              "messages": [{"role": "user", "content": f"How many letter r's are in '{word}'? Answer with just a number."}]})
    if r.status_code >= 400:
        raise requests.HTTPError(f"HTTP {r.status_code}: {r.text[:300]}", response=r)
    print("model says:", (r.json()["choices"][0]["message"]["content"] or "").strip())
except Exception as e:
    print("(API call failed:", type(e).__name__, str(e)[:300], ")")
print("code says: ", word.count("r"))

the model sees: ['st', 'raw', 'berry']
(API not configured: NotFoundError )
code says:  3


### D3.4b Let it think: reasoning on OpenRouter

Same question with `"reasoning": {"enabled": True}`: the model writes out its thinking before answering.
For the follow-up we send its `reasoning_details` back **unmodified**, so it continues its own reasoning.


In [6]:
import requests, json
base = os.environ.get("LLM_BASE_URL", "").rstrip("/")     # e.g. https://openrouter.ai/api/v1
url = base if base.endswith("/chat/completions") else base + "/chat/completions"
headers = auth_headers()
model = os.environ["LLM_MODEL"].split(",")[0].strip()
# OpenRouter: "reasoning": {...}. Gemini and other OpenAI-style endpoints: "reasoning_effort".
thinking = {"reasoning": {"enabled": True}} if "openrouter.ai" in base else {"reasoning_effort": "high"}
question = f"How many r's are in the word '{word}'?"

def post(body: dict) -> dict:
    r = requests.post(url, headers=headers, timeout=120, json=body)
    if r.status_code >= 400:                              # show the provider's reason, not just the status
        raise requests.HTTPError(f"HTTP {r.status_code}: {r.text[:400]}", response=r)
    return r.json()

try:
    # First call, with reasoning
    reply = post({"model": model, "messages": [{"role": "user", "content": question}], **thinking})
    message = reply["choices"][0]["message"]
    print("thinking:", (message.get("reasoning") or "(not returned by this model)")[:400])
    print("answer:  ", message.get("content"))
    usage = reply.get("usage") or {}
    print("reasoning tokens:", (usage.get("completion_tokens_details") or {}).get("reasoning_tokens"),
          "of", usage.get("completion_tokens"), "output tokens")

    # Follow-up: send the assistant message back, with reasoning_details unmodified when there are any
    previous = {"role": "assistant", "content": message.get("content")}
    if message.get("reasoning_details"):
        previous["reasoning_details"] = message["reasoning_details"]
    messages = [{"role": "user", "content": question}, previous,
                {"role": "user", "content": "Are you sure? Think carefully."}]
    second = post({"model": model, "messages": messages, **thinking})
    print("\nfollow-up:", second["choices"][0]["message"].get("content"))
except Exception as e:
    print("(API call failed:", type(e).__name__, str(e)[:400], ")")

thinking: The user is asking for the count of the letter 'r' in the word "strawberry".
I need to spell out the word and count the occurrences.
Word: s-t-r-a-w-b-e-r-r-y
1st 'r': after 't' (st**r**awberry)
2nd 'r': after 'e' (strawbe**r**ry)
3rd 'r': after the 2nd 'r' (strawber**r**y)
Total count: 3.
answer:   There are **3** r's in the word "strawberry".

Here is the breakdown:
st**r**awbe**rr**y
reasoning tokens: 74 of 141 output tokens

follow-up: Yes, I am sure. There are **3** r's in "strawberry".

**s-t-**`r`**-a-w-b-e-**`r`**`r`**-y**

1.  3rd letter: **r**
2.  8th letter: **r**
3.  9th letter: **r**


Point at: reasoning tokens are **output tokens**: you pay for them, and they count toward `max_tokens`
(a small `max_tokens` can leave an empty answer). With the OpenAI SDK the same option is
`extra_body={"reasoning": {"enabled": True}}`; in `llm_client` it is `LLMConfig(reasoning=...)`.


Lesson: counting and spelling belong in **code**, not in the model.

---
## D3.5 Does a whole paper fit?

In [ ]:
words_in_paper = 8_000                      # a typical AI paper
tokens_per_word = 1 / 0.75                  # rule of thumb for English
paper_tokens = int(words_in_paper * tokens_per_word)
price_in_per_1m = 0.20                      # GPT-5.6 Luna - in USD

print(f"~{paper_tokens:,} tokens for one paper")
for questions in (1, 10, 50):
    cost = paper_tokens * questions * price_in_per_1m / 1e6
    print(f"{questions:>3} questions, resending the whole paper each time: ~{paper_tokens * questions:>9,} input tokens  ≈ ${cost:.3f}")
print("\nThe model has no memory, so app resends the paper every call. Day 2: send only the relevant parts.")

~10,666 tokens for one paper
  1 questions, resending the whole paper each time: ~   10,666 input tokens  ≈ $0.002
 10 questions, resending the whole paper each time: ~  106,660 input tokens  ≈ $0.021
 50 questions, resending the whole paper each time: ~  533,300 input tokens  ≈ $0.107

The model has no memory, so a naive app resends the paper every call. Day 2: send only the relevant parts.
